# Sod shock tube

Runs the 1D Sod tube with one of warpSPH's compressible schemes and compares it with the exact Riemann solution, with the same functions the [case page](https://fluids.dev/cases/sod/) is built from. On a GPU it takes seconds. In Colab choose *Runtime > Change runtime type > GPU* first.

In [ ]:
# Setup: installs the latest warpSPH stack from GitHub when it is not already present.
import importlib.util, subprocess, sys
if importlib.util.find_spec('warpSPHCases') is None:
    stack = ['warpSPHCore', 'warpSPHIntegrators', 'warpSPHPlotting', 'warpSPH', 'warpSPHCases']
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q']
                          + [f'git+https://github.com/wi-re/{name}' for name in stack])
    # headless display for the vispy video (Colab has none)
    subprocess.call('apt-get -qq install -y xvfb > /dev/null', shell=True)
    from pyvirtualdisplay import Display
    Display(visible=0, size=(1280, 800)).start()

## Parameters

`scheme` is one of `CompSPH`, `Monaghan`, `CRKSPH`, `GSPH`, `InutsukaGSPH`, `MFM`, `MFV`. `nx` is the resolution (`dx = 1/nx`); the page uses 400.

In [ ]:
scheme = 'CompSPH'
nx = 400
video = True        # writes output.mp4 next to the trajectory; falls back to no video if the backend cannot start

## Run

In [ ]:
from warpSPHCases import quick

traj = quick.run('sod', scheme=scheme, nx=nx, video=video, spec={'exportInterval': 0.005})

## Result against the exact solution

In [ ]:
fig = quick.profiles(traj, 'sod', fields=['density', 'velocity', 'pressure', 'internalEnergy'])

In [ ]:
import pandas as pd
pd.DataFrame(quick.errors(traj, 'sod')).T

## Regression check

The page's reference run for this scheme at `nx = 400` is stored in the repository as a small golden file; a fresh run should agree within a relative L2 tolerance (bit equality only holds on the same GPU and driver).

In [ ]:
if nx == 400:
    quick.compareWithGolden(traj, 'compressible', 'sod', f'{scheme}__default__nx400')
else:
    print('golden files exist for nx = 400 only')

In [ ]:
# the run's video, when one was written
from pathlib import Path
from IPython.display import Video
mp4 = Path(traj.dir) / 'output.mp4'
Video(str(mp4), embed=True, width=640) if mp4.exists() else print('no video for this run')